# tcrkit — quick tour

One short example per capability. Every function takes one value or many (a list, dict
or DataFrame), returns a DataFrame with your input echoed back, and reports a bad row in
an `error` column instead of raising. For every option, see `docs/api.md`.

In [1]:
import pandas as pd

import tcrkit

## 1. `stitch` — V/J + CDR3 → a full-length chain

In [2]:
# Squeeze causes the output to be a pd.Series instead of a pd.DataFrame
tcrkit.stitch('TRBV19', 'TRBJ2-7', 'CASSIRSSYEQYF', chain='beta', squeeze=True)

sequence_aa         MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNL...
error                                                               -
stitchr_warnings    No valid leader region allele determined yet f...
Name: 0, dtype: str

A table works the same way. A `chain` column lets alpha and beta stitch together:

In [3]:
tcrs = pd.DataFrame({'v':     ['TRBV19',        'TRAV12-2'],
                     'j':     ['TRBJ2-7',       'TRAJ23'],
                     'cdr3':  ['CASSIRSSYEQYF', 'CAVNTGGGNKLTF'],
                     'chain': ['beta',          'alpha']})
tcrkit.stitch(tcrs, progress_bar=False)

,v,j,cdr3,chain,sequence_aa,error,stitchr_warnings
0,TRBV19,TRBJ2-7,CASSIRSSYEQYF,beta,MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNL...,-,No valid leader region allele determined yet f...
1,TRAV12-2,TRAJ23,CAVNTGGGNKLTF,alpha,MKSLRVLLVILWLQLSWVWSQQKEVEQNSGPLSVPEGAIASLNCTY...,-,No valid leader region allele determined yet f...


If your columns have different names, pass `col_mapping={'v': 'v_call', ...}`. Default specie is human, for mouse, pass `species='mouse'`.

## 2. `anarci` — a protein sequence → V/J genes + CDR loops

`allow=` limits ANARCI to the chain types you list. You don't need it in general, but
**if you are looking for alpha/beta chains, pass `allow={'A', 'B'}`.** Otherwise ANARCI
picks among every chain type it knows. Because alpha and delta overlap, it can call an
alpha chain delta:

In [4]:
alpha = ('AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMIFLIYQGSYDQQNATEGRYSLNFQKARKSANL'
         'VISASQLGDSAMYFCAMSQLNSGNTPLVFGKGTRLSVIA')
beta = ('MNAGVTQTPKFRVLKTGQSMTLLCAQDMNHDYMYWYRQDPGMGLRLIHYSVGEGTTAKGEVPDGYNVSRLKKQNFLL'
        'GLESAAPSQTSVYFCASSFTDTQYFGPGTRLTVL')

tcrkit.anarci(alpha) # ANARCI detects delta gene instead of alpha 

,sequence,CDR1_aa,CDR1_aa_start,CDR1_aa_end,CDR2_aa,CDR2_aa_start,CDR2_aa_end,CDR3_aa,CDR3_aa_start,CDR3_aa_end,query_start,query_end,species,chain_type,v_gene,v_identity,j_gene,j_identity,error
0,AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMI...,TSDPSYG,27,33,QGSYDQQN,51,58,AMSQLNSGNTPLV,93,105,0,114,human,D,TRAV14/DV4*01,0.979167,TRDJ4*01,0.642857,-


In [5]:
# Limiting the search space to only alpha and beta genes fixes the issue
pd.concat({'no allow':        tcrkit.anarci(alpha),
           "allow={'A','B'}": tcrkit.anarci(alpha, allow={'A', 'B'})}).droplevel(1)[
    ['chain_type', 'v_gene', 'j_gene', 'CDR3_aa']]

,chain_type,v_gene,j_gene,CDR3_aa
no allow,D,TRAV14/DV4*01,TRDJ4*01,AMSQLNSGNTPLV
"allow={'A','B'}",A,TRAV14/DV4*01,TRAJ29*01,AMSQLNSGNTPLV


In [6]:
# Several sequences can be processed at once
tcrkit.anarci({'alpha': alpha, 'beta': beta}, allow={'A', 'B'}, progress_bar=False)

,sequence,CDR1_aa,CDR1_aa_start,CDR1_aa_end,CDR2_aa,CDR2_aa_start,CDR2_aa_end,CDR3_aa,CDR3_aa_start,CDR3_aa_end,query_start,query_end,species,chain_type,v_gene,v_identity,j_gene,j_identity,error
alpha,AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMI...,TSDPSYG,27,33,QGSYDQQN,51,58,AMSQLNSGNTPLV,93,105,0,115,human,A,TRAV14/DV4*01,0.968750,TRAJ29*01,0.928571,-
beta,MNAGVTQTPKFRVLKTGQSMTLLCAQDMNHDYMYWYRQDPGMGLRL...,MNHDY,28,32,SVGEGT,50,55,ASSFTDTQY,93,101,1,111,human,B,TRBV6-2*01,0.978947,TRBJ2-3*01,1.000000,-


`collapse='region'` splits each sequence into its IMGT regions, FR1 to FR4, with their
positions and sequences. `per_residue=True` gives one row per residue instead.

In [7]:
tcrkit.anarci(alpha, allow={'A', 'B'}, collapse='region')

,region,start,end,length,imgt_start,imgt_end,sequence
0,FR1,1,26,26,1,26,AQKITQTQPGMFVQEKEAVTLDCTYD
0,CDR1,27,33,7,27,38,TSDPSYG
0,FR2,34,50,17,39,55,LFWYKQPSSGEMIFLIY
0,CDR2,51,58,8,56,65,QGSYDQQN
0,FR3,59,92,34,66,104,ATEGRYSLNFQKARKSANLVISASQLGDSAMYFC
0,CDR3,93,105,13,105,117,AMSQLNSGNTPLV
0,FR4,106,115,10,118,127,FGKGTRLSVI
0,,116,116,1,NaN,NaN,A


## 3. `add_cdr3_junctions` — make a CDR3 stitchable

A CDR3 without its conserved C104 / F118 will not stitch. `stitch` repairs nothing, so
restore the junction residues first. The J gene decides the last residue (F, W or C):

In [8]:
tcrkit.add_cdr3_junctions(cdr3=['ASSIRSSYEQY', 'AVMDSNYQLI'], j=['TRBJ2-7', 'TRAJ33'])

,cdr3,j,cdr3_fixed,changed,junction,purity,n,level,error
0,ASSIRSSYEQY,TRBJ2-7,CASSIRSSYEQYF,True,C/F,0.988983,291300,J,None
1,AVMDSNYQLI,TRAJ33,CAVMDSNYQLIW,True,C/W,1.000000,65517,J,None


Over a table it takes two steps, repair then stitch:

In [9]:
df = pd.DataFrame({'v': ['TRBV19', 'TRAV1-2'], 'j': ['TRBJ2-7', 'TRAJ33'],
                   'cdr3': ['ASSIRSSYEQY', 'AVMDSNYQLI'], 'chain': ['beta', 'alpha']})

df['cdr3'] = tcrkit.add_cdr3_junctions(df)['cdr3_fixed']
tcrkit.stitch(df, progress_bar=False)[['chain', 'cdr3', 'error']]

,chain,cdr3,error
0,beta,CASSIRSSYEQYF,-
1,alpha,CAVMDSNYQLIW,-


## 4. `normalize_tcr` / `normalize_mhc` — tidy gene and allele names

Different databases use slightly different notations of alleles and genes, here we have two functions that helps to normalize these notations. 

In [10]:
tcrkit.normalize_tcr(['TCRBV20S1', 'TRAV23/6', 'junk'], log_failures=False)

,symbol,standardized,species,error
0,TCRBV20S1,TRBV20-1,human,NaN
1,TRAV23/6,TRAV23/DV6,human,NaN
2,junk,NaN,NaN,not recognised by tidytcells


In [11]:
tcrkit.normalize_mhc(['A*0201', 'H2-Kb', 'DRB1*04:01', 'bad'], errors='coerce')

,allele_input,mhc_prefix,species,gene,allele,mhc_class,error
0,A*0201,HLA,human,A,A0201,I,NaN
1,H2-Kb,H2,mouse,K,H2-Kb,I,NaN
2,DRB1*04:01,HLA,human,DRB1,DRB1*0401,II,NaN
3,bad,NaN,NaN,NaN,NaN,NaN,ParseError: Could not parse 'bad'


## 5. `mhc_sequence` — an allele → its protein sequence

Spellings are normalized first, so `HLA-A*02:01` and `A*0201` find the same row. By
default you get the extracellular part:

In [12]:
tcrkit.mhc_sequence(['HLA-A*02:01', 'A*0201', 'DRB1*04:01'])

,allele_input,allele,sequence,length,gene,mhc_class,species,regions,signal_peptide,extracellular,transmembrane,cytoplasmic,error
0,HLA-A*02:01,A0201,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...,283,A,I,human,extracellular,None,1-283,None,None,None
1,A*0201,A0201,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...,283,A,I,human,extracellular,None,1-283,None,None,None
2,DRB1*04:01,DRB1*0401,GDTRPRFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGE...,198,DRB1,II,human,extracellular,None,1-198,None,None,None


With `squeeze=True`, one allele comes back as a Series instead of a one-row table, so
`r.sequence` or `r['length']` work directly:

In [13]:
r = tcrkit.mhc_sequence('HLA-A*02:01', squeeze=True)
r

allele                                                        A0201
sequence          GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...
length                                                          283
gene                                                              A
mhc_class                                                         I
species                                                       human
regions                                               extracellular
signal_peptide                                                 None
extracellular                                                 1-283
transmembrane                                                  None
cytoplasmic                                                    None
error                                                          None
Name: 0, dtype: object

The bundled sequences come from UniProt. `source='decodertcr'` uses DecoderTCR's HLA
reference instead: downloaded the first time, then read from a cache. A region means the
same stretch of the chain from either source, so here the two extracellular domains are
identical:

In [14]:
pd.concat({src: tcrkit.mhc_sequence(['HLA-A*02:01', 'DRB1*04:01'], source=src)
           for src in ('uniprot', 'decodertcr')})[['allele', 'length', 'sequence']]

allele  length  \
uniprot    0      A0201     283   
           1  DRB1*0401     198   
decodertcr 0      A0201     283   
           1  DRB1*0401     198   

                                                       sequence  
uniprot    0  GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...  
           1  GDTRPRFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGE...  
decodertcr 0  GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...  
           1  GDTRPRFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGE...

## 6. `annotate_mhc` — an MHC chain → its gene, IMGT numbering and helices

This is the MHC counterpart of `anarci`. Trust the `gene`. The `allele` is only the
closest of the 184 bundled alleles; `allele_identity` of 1.0 means an exact match.
`per_residue=True` gives one row per residue.

In [15]:
a2 = ('GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLG'
      'TLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLR'
      'AYLEGTCVEWLRRYLENGKETLQRT')                                  # HLA-A*02:01
b2m = ('IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYA'
       'CRVNHVTLSQPKIVKWDRDM')

tcrkit.annotate_mhc({'HLA-A2': a2, 'B2M': b2m})[
    ['gene', 'allele', 'allele_identity', 'mhc_class', 'domains', 'helix', 'error']]

,gene,allele,allele_identity,mhc_class,domains,helix,error
HLA-A2,A,A0201,1.0,I,"G-ALPHA1, G-ALPHA2","50-86, 140-176",None
B2M,B2M,B2M_human,1.0,-,NaN,NaN,None


`per_residue=True` gives one row per residue. `collapse=` merges them into segments: here
the strands (A–D), turns (AB, BC, CD) and helix of each groove domain.

In [16]:
tcrkit.annotate_mhc(a2, per_residue=True)

,position,aa,gene,allele,domain,imgt,region,helix,topology
0,1,G,A,A0201,G-ALPHA1,1,A,False,extracellular
0,2,S,A,A0201,G-ALPHA1,2,A,False,extracellular
0,3,H,A,A0201,G-ALPHA1,3,A,False,extracellular
0,4,S,A,A0201,G-ALPHA1,4,A,False,extracellular
0,5,M,A,A0201,G-ALPHA1,5,A,False,extracellular
...,...,...,...,...,...,...,...,...,...
0,178,T,A,A0201,G-ALPHA2,88,helix,False,extracellular
0,179,L,A,A0201,G-ALPHA2,89,helix,False,extracellular
0,180,Q,A,A0201,G-ALPHA2,90,helix,False,extracellular
0,181,R,A,A0201,G-ALPHA2,91,helix,False,extracellular


`print_residues` shows the same annotation under the sequence itself; `color=True` colours it by region. The `region` line's `H` is IMGT's whole helix region (50–92), the `helix` line the narrower docking-angle residues (50–86), so it sits inside it. Use the `helix` column as the reference for the groove helices:

In [17]:
tcrkit.print_residues(tcrkit.annotate_mhc({'HLA-A2': a2}, per_residue=True), width=70, color=True)

HLA-A2  182 residues
          1         11        21        31        41        51        61
sequence  GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAH
topology  OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
domain    1111111111111111111111111111111111111111111111111111111111111111111111
region    AAAAAAAAAAAAAA...BBBBBBBBBBB..CCCCCCCC...DDDDDDDDHHHHHHHHHHHHHHHHHHHHH
helix                                                      HHHHHHHHHHHHHHHHHHHHH

          71        81        91        101       111       121       131
sequence  SQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAA
topology  OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
domain    1111111111111111111122222222222222222222222222222222222222222222222222
region    HHHHHHHHHHHHHHHHHHHHAAAAAAAAAAAAAA...BBBBBBBBBBB..CCCCCCCC.DDDDDDDDHHH
helix     HHHHHHHHHHHHHHHH                                                     H

          141       

`imgt_regions` lists the IMGT ranges of the CDRs and groove helices, with their sources (`all_regions=True` adds every other region):

In [18]:
tcrkit.imgt_regions()

,scheme,mhc_class,chain,domain,region,column,imgt_start,imgt_end,skipped,source
0,tcr,-,-,V-DOMAIN,CDR1,region,27,38,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
1,tcr,-,-,V-DOMAIN,CDR2,region,56,65,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
2,tcr,-,-,V-DOMAIN,CDR3,region,105,117,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
3,mhc,I,heavy,G-ALPHA1,helix,helix,50,86,,tcrkit helix='imgt': the groove-helix residues...
4,mhc,I,heavy,G-ALPHA2,helix,helix,52,86,,tcrkit helix='imgt': the groove-helix residues...
5,mhc,II,alpha,G-ALPHA,helix,helix,50,86,,tcrkit helix='imgt': the groove-helix residues...
6,mhc,II,beta,G-BETA,helix,helix,52,86,"61B, 62",tcrkit helix='imgt': the groove-helix residues...


In [19]:
s = tcrkit.mhc_sequence(['HLA-A*02:01'], region='full', squeeze=True)['sequence']
tcrkit.annotate_mhc(s, collapse=['topology'])

,topology,start,end,length,imgt_start,imgt_end,sequence
0,signal_peptide,1,24,24,-24,-1,MAVMAPRTLVLLLSGALALTQTWA
0,extracellular,25,307,283,1,92,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...
0,transmembrane,308,332,25,NaN,NaN,IVGIIAGLVLFGAVITGAVVAAVMW
0,cytoplasmic,333,365,33,NaN,NaN,RRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV


## 7. `build_tcr_pmhc` — all of it over a table

This runs normalize → add junctions → stitch → MHC lookup → ANARCI, then checks that
ANARCI recovers the CDR3 that went in. Row `b` has a bad V gene and a serotype instead of
an allele. It is reported, and row `a` is still built:

In [20]:
raw = pd.DataFrame([
    {'id': 'a', 'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33', 'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'TCRBV20S1', 'beta_j': 'TRBJ2-7', 'beta_cdr3': 'ASSLGQAYEQY',
     'mhc_a': 'HLA-A*02:01'},
    {'id': 'b', 'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33', 'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'NOTAGENE', 'beta_j': 'TRBJ2-7', 'beta_cdr3': 'ASSLGQAYEQY',
     'mhc_a': 'A2'},
])

built = tcrkit.build_tcr_pmhc(raw, progress_bar=False)
built[['id', 'beta_v', 'mhc_a_allele', 'alpha_cdr3_ok', 'beta_cdr3_ok', 'ok', 'error']]

,id,beta_v,mhc_a_allele,alpha_cdr3_ok,beta_cdr3_ok,ok,error
0,a,TRBV20-1,A0201,True,True,True,-
1,b,NOTAGENE,A2,True,NaN,False,beta V call not recognised | beta did not stit...


Next: `tcrkit_metrics.ipynb` for repertoire metrics, `docs/api.md` for every option, and
`tcrkit --help` for the command line.